## Example 1 — Config: `thread_id` identifies a conversation

**Definition:** `config` supplies execution settings. When a checkpointer is enabled, `thread_id` identifies which conversation's saved state to resume.

We'll use the same thread ID across three turns so the agent can use its prior message history.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent_config_demo = create_agent(
    model="openai:gpt-5.5",
    checkpointer=InMemorySaver(),
)

config = {
    "configurable": {
        "thread_id": "conversation-001"
    }
}

# Turn 1
result1 = agent_config_demo.invoke(
    {"messages": [{"role": "user", "content": "My name is Alex."}]},
    config=config,
)
print("Turn 1:", result1["messages"][-1].content)

# Turn 2: same thread, so previous conversation history is available
result2 = agent_config_demo.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config,
)
print("Turn 2:", result2["messages"][-1].content)

# Turn 3: same thread again
result3 = agent_config_demo.invoke(
    {"messages": [{"role": "user", "content": "What did I tell you earlier?"}]},
    config=config,
)
print("Turn 3:", result3["messages"][-1].content)

### What to observe

- All three invocations reuse `thread_id="conversation-001"`.
- The checkpointer saves and resumes state for that thread.
- A different thread ID represents a different conversation state.
- `thread_id` does not identify the user; it identifies the conversation thread.

## Example 2 — State: custom data maintained across turns

**Definition:** `state` is the data the agent manages, including conversation messages and custom fields. With a checkpointer, custom state can carry forward across invocations on the same thread.

Here, middleware increments `call_count` after each model response.

In [3]:
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver


class MyState(AgentState):
    call_count: int


@after_model
def increment_counter(state: MyState, runtime):
    return {
        "call_count": state.get("call_count", 0) + 1
    }


agent_state_demo = create_agent(
    model="openai:gpt-5.5",
    state_schema=MyState,
    middleware=[increment_counter],
    checkpointer=InMemorySaver(),
)

config = {
    "configurable": {
        "thread_id": "state-thread-001"
    }
}

# Turn 1: initialize custom state
state_result1 = agent_state_demo.invoke(
    {
        "messages": [{"role": "user", "content": "Hello!"}],
        "call_count": 0,
    },
    config=config,
)
print("Turn 1 counter:", state_result1["call_count"])

# Turn 2: send only the new message; do not reset call_count
state_result2 = agent_state_demo.invoke(
    {"messages": [{"role": "user", "content": "Tell me about Python."}]},
    config=config,
)
print("Turn 2 counter:", state_result2["call_count"])

# Turn 3: continue the same saved state
state_result3 = agent_state_demo.invoke(
    {"messages": [{"role": "user", "content": "What is a variable?"}]},
    config=config,
)
print("Turn 3 counter:", state_result3["call_count"])

Turn 1 counter: 1
Turn 2 counter: 2
Turn 3 counter: 3


### What to observe

The counter should progress from 1 to 2 to 3, assuming the middleware runs once per model response.

- `MyState` declares the `call_count` field.
- The first invocation initializes it to `0`.
- Middleware returns an update; LangGraph merges it into the current state.
- The checkpointer allows the updated value to be restored on the next turn.

This example counts model responses processed by this middleware, not tool calls in general.

## Example 3 — Context: per-invocation information

**Definition:** `context` supplies application data and dependencies to the current invocation. It is not automatically persisted as agent state, so pass it again on each turn when needed.

This example passes the user's ID and subscription tier through context. Middleware reads them using `runtime.context`.

In [4]:
from dataclasses import dataclass

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import before_model
from langgraph.checkpoint.memory import InMemorySaver


class ContextDemoState(AgentState):
    pass


@dataclass
class UserContext:
    user_id: str
    subscription_tier: str


@before_model
def inspect_context(state: ContextDemoState, runtime):
    print("User ID:", runtime.context.user_id)
    print("Subscription:", runtime.context.subscription_tier)
    return None


agent_context_demo = create_agent(
    model="openai:gpt-5.5",
    state_schema=ContextDemoState,
    context_schema=UserContext,
    middleware=[inspect_context],
    checkpointer=InMemorySaver(),
)

config = {
    "configurable": {
        "thread_id": "context-thread-001"
    }
}

# Turn 1: provide context
context_result1 = agent_context_demo.invoke(
    {"messages": [{"role": "user", "content": "Hello!"}]},
    config=config,
    context=UserContext(
        user_id="user-123",
        subscription_tier="premium",
    ),
)

# Turn 2: same thread, context is supplied again
context_result2 = agent_context_demo.invoke(
    {"messages": [{"role": "user", "content": "What can you help me with?"}]},
    config=config,
    context=UserContext(
        user_id="user-1234",
        subscription_tier="premium",
    ),
)

# Turn 3: same thread, context is supplied again
context_result3 = agent_context_demo.invoke(
    {"messages": [{"role": "user", "content": "Thank you!"}]},
    config=config,
    context=UserContext(
        user_id="user-12345",
        subscription_tier="premium",
    ),
)

User ID: user-123
Subscription: premium
User ID: user-1234
Subscription: premium
User ID: user-12345
Subscription: premium


### What to observe

- The conversation history persists through the checkpointer.
- Context is supplied separately for each invocation.
- `runtime.context.user_id` reads the current invocation's user ID.
- Context does not automatically appear in `result` or become a saved state field.

In a real application, derive the user ID from trusted authentication rather than blindly trusting a client-provided identity.

In [7]:
from dataclasses import dataclass
from typing import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import before_model


# 1. Define a function
def get_user_name(user_id: str) -> str:
    users = {
        "user-123": "John Smith",
        "user-456": "Alice Johnson",
    }
    return users.get(user_id, "Unknown User")


# 2. Define the context schema
@dataclass
class Context:
    user_id: str
    get_user_name: Callable[[str], str]


# 3. Create middleware that accesses the function
@before_model
def identify_user(state, runtime):
    user_id = runtime.context.user_id

    # Call the function provided through context
    name = runtime.context.get_user_name(user_id)

    print(f"Current user: {name}")

    return None


# 4. Create the agent
agent = create_agent(
    model="openai:gpt-5.5",
    tools=[],
    context_schema=Context,
    middleware=[identify_user],
)


# 5. Invoke the agent and supply context
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Hello!"}
        ]
    },
    context=Context(
        user_id="user-4564",
        get_user_name=get_user_name,
    ),
)

print("\nAgent response:")
print(result["messages"][-1].content)

Current user: Unknown User

Agent response:
Hello! How can I help you today?


## Final comparison

| Feature | Config | State | Context |
|---|---|---|---|
| Main purpose | Control execution | Maintain agent data | Supply per-run data/dependencies |
| Example | `thread_id` | `call_count` | `user_id` |
| Passed through | `config=` | Input dictionary | `context=` |
| Carries forward automatically? | Reuse config as needed | Yes, with a checkpointer and same thread | No; supply it again when needed |
| Automatically stored as agent state? | No | State is checkpointed when configured | No |

### Mental model

- **Config:** How should this execution run, and which thread should be resumed?
- **State:** What data is the agent maintaining and updating?
- **Context:** What application information or dependencies does this invocation need?

**Version note:** LangChain's APIs can evolve. If a hook signature differs in your installed version, check the current LangChain middleware documentation and package version.


State: State stores the data an agent needs to track, remember, and update as it progresses through a workflow, such as messages, user input, database query results, order status, or retry count. This data can be updated during execution and persisted using a checkpointer when needed.

Context: Context provides the information, resources, and dependencies that the application makes available to an agent during execution, such as a database service, API client, authenticated user, or configuration. Middleware and tools can access it through `runtime.context` to perform operations without storing live resources in the agent's state.